# H&M 마지막 주 test — seed49, M1·M4-B·M5-B
**L4 GPU에서 이 노트북 하나만 실행합니다. 세 모형을 순차 학습하며 다른 시드로 자동 확대하지 않습니다.**

- 학습: **2018-09-20~2020-09-15 전체**. test: **2020-09-16~22**. validation·holdout 없음.
- 각 모형 **300epoch 고정**, 마지막 checkpoint만 test 평가. 조기종료·test 최고점 선택 없음.
- 그래프/BPR은 학습 전체 기간, CLV 관측창은 학습 종료 전365일. 신규상품 추천으로 train pair는 정답·후보에서 제외, MIN_ITEM_INTER=1, uniform K=1.
- M1=이진 LightGCN/plain BPR. M4-B=이진 LightGCN/N·V 분리 행가중. M5-B=중심화 N·V 그래프/**동일한 M4-B 가중치**. M2·M5-A 추가 학습 없음.
- B 가중치: raw=(1+0.5q_N)·(1+0.5q_V·상품가격백분위·경제구간적합도), 학습행 평균으로 정규화. 기존 H&M 구현대로 CLV 무효 고객 N/V=0; 추가 경제정보 유효성 마스크는 쓰지 않습니다. 두 가중 모형의 가중치 해시를 함께 저장합니다.
- M3는 N/V 두 축 모두 포함, 학습구간에서 가중치 CV=.20을 목표로 β를 보정하고 가격·인기·연결수 대리변수 검사 후 학습합니다. 기존 개발구간 β/평균가중치를 강제로 재사용하지 않습니다.
- ID64/2층/batch131072/lr=.0005/L2=.001. 각 모형은 단일 optimizer로 처음부터 학습합니다.
- 기존 다른 분할 결과/checkpoint는 재사용하지 않습니다. 이번 실행의 완료 결과와 optimizer·난수상태 포함 epoch checkpoint만 자동 재개합니다.

## 사전 판독
전체 사용자 가격·구매금액 가중 적중값@10과 가중 NDCG@10이 **각각 M1 및 M4-B보다 높고**, 여섯 Recall/NDCG@10·20·50이 **각각 M1의99% 이상**인지 확인합니다. M4-B 대비 정확도 보호 여부도 별도로 보고합니다. 모든 세그먼트·노출·불리한 결과를 저장합니다.

M5-B−M4-B는 같은 손실에서 **M3 추가효과**를 묻습니다. M3 단독/순열 arm이 없으므로 완전한 상호작용·CLV 배정귀속을 입증하지 않습니다. 단일 시드로 안정성·유의성을 주장하지 않습니다. 마지막 주는 과거 연구에서 노출된 test이므로 새 독립 확증이 아니며, 결과를 보고 같은 test에 맞춰 설정을 재선택하지 않습니다.

소형 CPU 분할·학습/재개·동일 가중치·보고 검사 통과. 실제 H&M 전체 데이터/L4 학습은 미실행입니다.


## 1. Drive 연결·고정 코드 로드
런타임 → 런타임 유형 변경 → L4 GPU. Drive 연결 오류가 나면 연결 복구 후 이 셀부터 다시 실행하세요.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json, torch
SOURCE_COMMIT = 'fd23432b0930c40c9633920a4223502bf3df064c'
REPO = Path('/content/clv-hm-m4b-m5b-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
assert 'L4' in torch.cuda.get_device_name(0), '런타임 유형을 L4로 변경하세요.'
sys.path.insert(0, str(REPO))
print('source:', SOURCE_COMMIT, '| GPU:', torch.cuda.get_device_name(0))


## 2. 새 분할·가중치 준비 — 학습 없음
학습 시작 전에 날짜 경계·세 모형 범위·N/V 그래프 검사를 수행합니다.

In [ ]:
import clv_m2_m5_lastweek_test10 as screen
cfg = screen.configure('hm', seeds=(49,), experiment='hm_m4b_m5b')
assert cfg.dataset == 'hm' and cfg.seeds == (49,) and cfg.epochs == 300
assert screen.models_for(cfg) == (screen.M1, screen.M4_B, screen.BASE)
prepared = screen.prepare(cfg)
assert prepared['protocol']['total_fits'] == 3
assert set(prepared['data']['splits']) == {'test'}
print('자동재개 경로:', prepared['run_dir'])


## 3. M1 → M4-B → M5-B 순차 학습
총3회×300epoch입니다. 매epoch 로그와 optimizer·난수상태를 저장합니다. 중단 후 같은 노트북을 재실행하면 마지막 저장 epoch에서 재개하고 완료 모형은 건너뜁니다. 중간 test 평가는 하지 않습니다.

In [ ]:
result = screen.run(cfg, prepared)
assert len(result['absolute']) == 3 and result['reading']['seed_count'] == 1
assert result['reading']['evaluated_epoch'] == 300
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))


## 4. 전체 사용자 성능과 비교
@10·20·50을 모두 표시합니다. 전체·저/중/고CLV·노출의 전체 원본은 ZIP에 포함합니다.

In [ ]:
metrics = [f'{m}@{k}' for k in (10,20,50) for m in ('recall','ndcg','price_purchase_amount_weighted_hit','vndcg')]
labels = {screen.M1:'M1', screen.M4_B:'M4-B', screen.BASE:'M5-B (M3+M4-B)'}
print(result['absolute'].set_index('model_id')[metrics].T.rename(columns=labels).to_string())
comparison = result['comparison'].copy()
comparison['model_id'] = comparison.model_id.map(labels)
comparison['reference'] = comparison.reference.map(labels)
print('\n상대변화율(%)')
print(comparison[comparison.metric.isin(metrics)].pivot(index='metric', columns=['model_id','reference'], values='relative_change_pct').to_string())
print('\n그래프·손실 동일성 진단')
print(result['diagnostics'].to_string(index=False))
print(json.dumps(result['paths'], ensure_ascii=False, indent=2))


## 5. 결과 ZIP 다운로드
absolute/summary/comparison/paired_comparison/diagnostics/result.json/protocol.json 포함. 학습이력은 result.json, checkpoint는 Drive에 남습니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
zip_path = Path('/content/hm_m4b_m5b_lastweek_seed49_results.zip')
with ZipFile(zip_path, 'w', ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
    archive.write(prepared['run_dir']/'protocol.json', arcname='protocol.json')
files.download(str(zip_path))
